In [5]:
!py -3.11 --version

Python 3.11.0


In [2]:
!pip install scikit-learn pandas matplotlib opencv-python mediapipe seaborn

  Obtaining dependency information for seaborn from https://files.pythonhosted.org/packages/83/11/00d3c3dfc25ad54e731d91449895a79e4bf2384dc3ac01809010ba88f6d5/seaborn-0.13.2-py3-none-any.whl.metadata
   ---------------------------------------- 0.0/294.9 kB ? eta -:--:--
   ---------------------------------------- 0.0/294.9 kB ? eta -:--:--
   - -------------------------------------- 10.2/294.9 kB ? eta -:--:--
   - -------------------------------------- 10.2/294.9 kB ? eta -:--:--
   - -------------------------------------- 10.2/294.9 kB ? eta -:--:--
   ---- ---------------------------------- 30.7/294.9 kB 187.9 kB/s eta 0:00:02
   ---- ---------------------------------- 30.7/294.9 kB 187.9 kB/s eta 0:00:02
   ----- --------------------------------- 41.0/294.9 kB 140.9 kB/s eta 0:00:02
   ----- --------------------------------- 41.0/294.9 kB 140.9 kB/s eta 0:00:02
   -------- ------------------------------ 61.4/294.9 kB 173.0 kB/s eta 0:00:02
   ---------- ----------------------------


[notice] A new release of pip is available: 23.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
!pip install scikit-learn pandas matplotlib opencv-python mediapipe seaborn joblib


[notice] A new release of pip is available: 23.2.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
!py -3.11 -m pip install scikit-learn pandas matplotlib opencv-python mediapipe seaborn joblib


[notice] A new release of pip available: 22.3 -> 26.3
[notice] To update, run: C:\Users\PMLS\AppData\Local\Programs\Python\Python311\python.exe -m pip install --upgrade pip


In [2]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

# 1. Load the preprocessed sequences
X = np.load('X_train_video_data.npy')
y = np.load('y_train_video_labels.npy')

# 2. Encode labels
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
y_categorical = to_categorical(y_encoded)

# 3. Split (We are splitting our train folder into 80/20 for validation)
X_train, X_test, y_train, y_test = train_test_split(X, y_categorical, test_size=0.2, random_state=42)

print(f"Training sequences: {len(X_train)}")
print(f"Testing sequences: {len(X_test)}")

# 4. Build the LSTM Model
model = Sequential()
# LSTM Layer 1
model.add(LSTM(64, return_sequences=True, activation='relu', input_shape=(30, 63)))
model.add(Dropout(0.2))
# LSTM Layer 2
model.add(LSTM(128, return_sequences=False, activation='relu'))
model.add(Dropout(0.2))
# Dense Layers
model.add(Dense(64, activation='relu'))
model.add(Dense(32, activation='relu'))
# Output Layer (Number of words)
model.add(Dense(len(label_encoder.classes_), activation='softmax')) 

model.compile(optimizer='Adam', loss='categorical_crossentropy', metrics=['categorical_accuracy'])
model.summary()

# 5. Train
print("Training LSTM... (This may take a few minutes)")
history = model.fit(X_train, y_train, epochs=50, validation_data=(X_test, y_test))

# 6. Save
model.save('lstm_word_model.h5')
print("Model saved as 'lstm_word_model.h5'")

# Save the label encoder so we can use it in the webcam test
import joblib
joblib.dump(label_encoder, 'lstm_label_encoder.pkl')
print("Label encoder saved.")

Training sequences: 1141
Testing sequences: 286


C:\Users\PMLS\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                          │ (None, 30, 64)              │          32,768 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout (Dropout)                    │ (None, 30, 64)              │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ lstm_1 (LSTM)                        │ (None, 128)                 │          98,816 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_1 (Dropout)                  │ (None, 128)                 │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense (Dense)                        │ (None, 64)                  │           8,256 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_1 (Dense)                      │ (None, 32)                  │           2,080 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_2 (Dense)                      │ (None, 99)                  │           3,267 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 145,187 (567.14 KB)

 Trainable params: 145,187 (567.14 KB)

 Non-trainable params: 0 (0.00 B)

Training LSTM... (This may take a few minutes)
Epoch 1/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 12s 119ms/step - categorical_accuracy: 0.0131 - loss: 4.6202 - val_categorical_accuracy: 0.0140 - val_loss: 4.5620
Epoch 2/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 2s 68ms/step - categorical_accuracy: 0.0237 - loss: 4.5660 - val_categorical_accuracy: 0.0315 - val_loss: 4.5640
Epoch 3/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 2s 65ms/step - categorical_accuracy: 0.0324 - loss: 4.5274 - val_categorical_accuracy: 0.0210 - val_loss: 4.5029
Epoch 4/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 3s 78ms/step - categorical_accuracy: 0.0228 - loss: 4.4361 - val_categorical_accuracy: 0.0245 - val_loss: 4.4097
Epoch 5/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 3s 76ms/step - categorical_accuracy: 0.0316 - loss: 4.2658 - val_categorical_accuracy: 0.0385 - val_loss: 4.2581
Epoch 6/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 3s 74ms/step - categorical_accuracy: 0.0438 - loss: 4.1384 - val_categorical_accuracy: 0.0350 - val_loss: 4.1813
Epoch 7/50
36/36 ━━━━━━━━━━━━━━━━━━━━ 3s 79ms

Model saved as 'lstm_word_model.h5'
Label encoder saved.


In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from tensorflow.keras.utils import to_categorical

# 1. Load data
X = np.load('X_train_video_data.npy') # Shape: (1141, 10, 63)
y = np.load('y_train_video_labels.npy')

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
y_categorical = to_categorical(y_encoded)

X_train, X_test, y_train, y_test = train_test_split(X, y_categorical, test_size=0.2, random_state=42)

# 2. Build a lighter, more robust LSTM
model = Sequential()
# Use Input layer instead of input_shape
model.add(Input(shape=(10, 63))) 
model.add(LSTM(32, return_sequences=True, activation='relu')) # Reduced from 64
model.add(Dropout(0.3)) # Increased dropout
model.add(LSTM(64, return_sequences=False, activation='relu')) # Reduced from 128
model.add(Dropout(0.3))
model.add(Dense(32, activation='relu'))
model.add(Dense(len(label_encoder.classes_), activation='softmax')) 

model.compile(optimizer='Adam', loss='categorical_crossentropy', metrics=['categorical_accuracy'])

# 3. Early Stopping (Stops training if val_accuracy stops improving)
early_stop = EarlyStopping(monitor='val_categorical_accuracy', patience=10, restore_best_weights=True, mode='max')

print("Training Full Dataset LSTM...")
history = model.fit(
    X_train, y_train, 
    epochs=100, # Let it run up to 100, but early stopping will cut it off
    validation_data=(X_test, y_test),
    callbacks=[early_stop]
)

model.save('lstm_full_dataset_model.h5')
print("Model saved!")